# 03. Inventory Optimization, Safety Stock, & Stockout Risk Flagging
## Supply Chain Demand Forecasting & Inventory Optimization Project

### Executive & Business Objective
> **Target Business Question:**
> *"For our top 20 SKUs by revenue, what should the reorder point and safety stock be for the next quarter, given demand variability and current supplier lead times — and which SKUs are at highest stockout risk?"*

In this notebook, we execute the core mathematical operations of the project:
1. Calculate **Safety Stock (SS)** at a 95% cycle service level factor ($Z = 1.65$).
2. Compute **Reorder Point (ROP)** combining lead-time forecast demand and safety stock.
3. Compute the **Economic Order Quantity (EOQ)** to balance purchase order costs against annual holding costs.
4. Compare current warehouse inventory against ROP to flag **High Risk**, **Medium Risk**, and **Healthy** items.
5. Rank at-risk SKUs by total **financial revenue impact** to direct immediate supply chain action.


In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

reorder_df = pd.read_csv(os.path.join(OUTPUTS_DIR, "reorder_recommendations.csv"))
print(f"Loaded reorder recommendations for {len(reorder_df)} Top SKUs.")
reorder_df.head()


### 1. Mathematical Formulas & Inventory Parameters
The inventory optimization engine applies the standard industrial operations research equations:

1. **Safety Stock ($SS$):**
   $$SS = Z \times \sigma_{\text{demand}} \times \sqrt{L_{\text{weeks}}}$$
   - $Z = 1.65$: Standard normal quantile for a **95% cycle service level**
   - $\sigma_{\text{demand}}$: Standard deviation of weekly demand
   - $L_{\text{weeks}}$: Supplier lead time in weeks ($\text{lead\_time\_days} / 7$)

2. **Reorder Point ($ROP$):**
   $$ROP = (\bar{d}_{\text{forecast}} \times L_{\text{weeks}}) + SS$$
   When on-hand inventory drops to or below $ROP$, a purchase order must be placed immediately.

3. **Economic Order Quantity ($EOQ$):**
   $$EOQ = \sqrt{\frac{2 \times D_{\text{annual}} \times S}{H}}$$
   - $D_{\text{annual}} = \bar{d}_{\text{forecast}} \times 52$: Annualized demand
   - $S$: Fixed administrative cost per purchase order
   - $H = \text{holding\_rate} \times \text{unit\_cost}$: Annual holding cost per unit


In [2]:
params_table = reorder_df[['sku_id', 'product_name', 'lead_time_days', 'avg_forecast_weekly', 'demand_stddev_weekly', 'safety_stock', 'reorder_point', 'current_stock', 'eoq', 'risk_status']].head(10)
params_table


### 2. Stockout Risk Classification
We classify each SKU into three actionable tiers:
- **HIGH RISK (Critical Deficit):** $\text{Current Stock} < ROP$. Inventory is insufficient to cover expected demand during supplier lead time.
- **MEDIUM RISK (Approaching ROP):** $ROP \le \text{Current Stock} < ROP + 0.5 \times SS$. Item is within half a safety stock of triggering an order.
- **HEALTHY:** $\text{Current Stock} \ge ROP + 0.5 \times SS$. Sufficient buffer stock exists.


In [3]:
risk_counts = reorder_df['risk_status'].value_counts()
print("Stockout Risk Distribution:")
for status, count in risk_counts.items():
    print(f" - {status}: {count} SKUs ({count/len(reorder_df)*100:.1f}%)")

colors = {'HIGH RISK': '#C00000', 'MEDIUM RISK': '#FFC000', 'HEALTHY': '#70AD47'}
plt.figure(figsize=(7, 7))
plt.pie(risk_counts, labels=risk_counts.index, autopct='%1.1f%%', 
        colors=[colors.get(k, '#888888') for k in risk_counts.index], 
        startangle=140, explode=[0.04] * len(risk_counts),
        textprops={'fontsize': 12, 'fontweight': 'bold'})
plt.title('Top 20 SKUs: Inventory Stockout Risk Distribution', fontsize=14, fontweight='bold')
plt.show()


### 3. Financial Revenue at Risk
For any SKU where $\text{Current Stock} < ROP$, the immediate unit deficit is:
$$\text{Deficit Units} = ROP - \text{Current Stock}$$
$$\text{Revenue at Risk} = \text{Deficit Units} \times \text{Unit Selling Price}$$

This directly quantifies the financial loss if replenishment orders are not expedited.


In [4]:
high_risk = reorder_df[reorder_df['risk_status'] == 'HIGH RISK'].sort_values('revenue_at_risk', ascending=False)
total_at_risk_revenue = high_risk['revenue_at_risk'].sum()

print(f"Total High Risk SKUs: {len(high_risk)}")
print(f"Total Immediate Revenue at Risk: ${total_at_risk_revenue:,.2f}\n")

high_risk[['sku_id', 'product_name', 'category', 'unit_price', 'current_stock', 'reorder_point', 'units_at_risk', 'revenue_at_risk', 'recommended_order_qty']]


### 4. Ranking High Risk SKUs by Revenue Exposure
Let's visualize the top revenue exposure items to give executive leadership clear visibility into where capital must be immediately allocated.


In [5]:
plt.figure(figsize=(12, 6))
bars = plt.barh(high_risk['product_name'], high_risk['revenue_at_risk'] / 1000.0, color='#C00000', edgecolor='darkred', alpha=0.85)
plt.xlabel('Revenue at Risk ($ Thousands)', fontsize=12, fontweight='bold')
plt.ylabel('Product Name', fontsize=12, fontweight='bold')
plt.title('High-Risk SKUs Ranked by Revenue Exposure ($k)', fontsize=14, fontweight='bold', pad=15)
plt.gca().invert_yaxis()

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.8, bar.get_y() + bar.get_height()/2, f"${w:.1f}k", 
             va='center', ha='left', fontsize=10, fontweight='bold', color='#2F3542')

plt.xlim(0, max(high_risk['revenue_at_risk']/1000.0) * 1.18)
plt.tight_layout()
plt.show()


### 5. Inventory vs. Reorder Point Comparison
Let's compare current stock levels against Reorder Points and Safety Stock thresholds across all Top 20 items.


In [6]:
plt.figure(figsize=(15, 7))
x = np.arange(len(reorder_df))
width = 0.35

plt.bar(x - width/2, reorder_df['current_stock'], width, label='Current On-Hand Stock', color='#2E75B6')
plt.bar(x + width/2, reorder_df['reorder_point'], width, label='Reorder Point (ROP)', color='#ED7D31')

plt.xlabel('Top 20 SKUs (Ranked by Revenue)', fontsize=11, fontweight='bold')
plt.ylabel('Units', fontsize=11, fontweight='bold')
plt.title('Current Stock vs. Reorder Point (ROP) Across Top 20 SKUs', fontsize=14, fontweight='bold', pad=15)
plt.xticks(x, reorder_df['sku_id'], rotation=45)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


### Executive Summary & Operational Recommendations
1. **Immediate Purchase Orders Required:**
   - **8 of the top 20 SKUs** are currently below their calculated Reorder Point.
   - Total immediate revenue exposed to stockout risk is **$211,727.80**.
   - Top 3 revenue exposure drivers:
     - `SKU-004` (Dual-Motor Electric Standing Desk): **$47,904.00** at risk
     - `SKU-001` (Pro ANC Headphones): **$47,878.29** at risk
     - `SKU-011` (Custom Mechanical Gaming Keyboard): **$46,237.28** at risk
2. **Recommended Order Execution:**
   - Expedite purchase orders totaling **1,461 units** across the 8 high-risk SKUs using our EOQ-adjusted quantities.
3. **Supplier Collaboration:**
   - For `SUP-104` (Shenzhen Global Direct) with a 28-day lead time, negotiate split air-freight shipments for `SKU-011` to prevent immediate stock depletion.
